In [0]:
from pyspark.sql import SparkSession

data = [
    (1,  "C001", "Ravi",   "IT",       80000, "2026-01-10", 100,  "ACTIVE"),
    (2,  "C001", "Ravi",   "IT",       90000, "2026-02-15", 200,  "ACTIVE"),
    (3,  "C001", "Ravi",   "IT",       90000, "2026-03-20", 150,  "ACTIVE"),
    (4,  "C001", "Ravi",   "IT",       70000, "2026-04-10", 300,  "ACTIVE"),

    (5,  "C002", "Suresh", "IT",       95000, "2026-01-05", 500,  "ACTIVE"),
    (6,  "C002", "Suresh", "IT",       95000, "2026-02-10", 100,  "ACTIVE"),
    (7,  "C002", "Suresh", "IT",       85000, "2026-03-15", 250,  "ACTIVE"),

    (8,  "C003", "Priya",  "HR",       65000, "2026-01-12", 400,  "ACTIVE"),
    (9,  "C003", "Priya",  "HR",       70000, "2026-02-18", 150,  "ACTIVE"),
    (10, "C003", "Priya",  "HR",       70000, "2026-03-25", 350,  "ACTIVE"),

    (11, "C004", "Anita",  "HR",       75000, "2026-01-20", 600,  "ACTIVE"),
    (12, "C004", "Anita",  "HR",       60000, "2026-02-22", 200,  "INACTIVE"),

    (13, "C005", "John",   "FINANCE",  80000, "2026-01-08", 700,  "ACTIVE"),
    (14, "C005", "John",   "FINANCE",  85000, "2026-02-14", 300,  "ACTIVE"),
    (15, "C005", "John",   "FINANCE",  85000, "2026-03-19", 500,  "ACTIVE"),

    (16, "C006", "David",  "FINANCE",  90000, "2026-01-15", 1000, "ACTIVE"),
    (17, "C006", "David",  "FINANCE",  70000, "2026-02-20", 400,  "ACTIVE"),

    # Duplicate records
    (18, "C007", "Kiran",  "SALES",    55000, "2026-01-10", 200,  "ACTIVE"),
    (19, "C007", "Kiran",  "SALES",    60000, "2026-02-10", 300,  "ACTIVE"),
    (20, "C007", "Kiran",  "SALES",    60000, "2026-02-10", 300,  "ACTIVE"),

    (21, "C008", "Meena",  "SALES",    65000, "2026-01-05", 450,  "ACTIVE"),
    (22, "C008", "Meena",  "SALES",    70000, "2026-02-05", 550,  "ACTIVE"),

    # NULL scenarios
    (23, None,   "Arun",   "IT",       72000, "2026-04-01", 250,  "ACTIVE"),
    (24, "C009", None,     "IT",       68000, "2026-04-05", 300,  "ACTIVE"),
    (25, "C010", "Raj",    None,       75000, "2026-04-10", 400,  "ACTIVE"),
    (26, "C011", "Sneha",  "HR",       None,  "2026-04-15", 350,  "ACTIVE"),
    (27, "C012", "Vijay",  "FINANCE",  82000, None,         500,  "ACTIVE"),
    (28, "C013", "Kavya",  "SALES",    62000, "2026-04-20", None, "ACTIVE"),
    (29, "C014", "Manoj",  "IT",       None,  None,         None, None),
    (30, None,   None,     None,       None,  None,         None, None)
]

columns = [
    "id",
    "customerID",
    "name",
    "department",
    "salary",
    "orderDate",
    "amount",
    "status"
]

df = spark.createDataFrame(data, columns)
display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# # Delete duplicate records
df_removedupli = df.dropDuplicates(subset=["customerID"])
# display(df_removedupli)

#Avg Salay based on each department
wDept = Window.partitionBy("department")
df_AvgSalByDept = df_removedupli.select("department","salary").withColumn("AvgSalary", F.sum("salary").over(wDept)).orderBy("ID")
display(df_AvgSalByDept)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_RemoveNullRows = df.dropna(subset=["customerID","salary"])
window_spec = Window.partitionBy("customerID")
wDept = Window.partitionBy("department").orderBy(F.col("orderDate").desc())

#latest Transaction by department
df_data = df.createOrReplaceTempView("dfOrginalData")
# df_latestTrans = spark.sql \
#             ("""SELECT *
#     FROM dfOrginalData
#     QUALIFY ROW_NUMBER() OVER (
#         PARTITION BY department
#         ORDER BY OrderDate DESC
#     ) = 1
#              """)

df_latestTrans = df.withColumn("rn", F.row_number().over(wDept)).filter(F.col("rn")==1).drop("rn")
#display(df_latestTrans)

#Find duplicate records
df_result = df_RemoveNullRows.withColumn("duplicate_count", F.count("*").over(window_spec)).filter(F.col("duplicate_count")>1)
#display(df_result)

# SQL query to get only duplicate records (by customerID)
df_duprecords = spark.sql("""
    SELECT *
    FROM dfOrginalData
    WHERE customerID IN 
    (
        SELECT customerID
        FROM dfOrginalData
        WHERE customerID IS NOT NULL
        GROUP BY customerID
        HAVING COUNT(*) > 1
    )
""")
display(df_duprecords)



# df_CountOfDuplicateCustID = df_RemoveNullRows.withColumn("cusIDCount", F.count("*").over(window_spec))
# # display(df_CountOfDuplicateCustID)

# # df_GetNameByDept = df_CountOfDuplicateCustID.groupBy("department").agg(F.max("cusIDCount").alias("maxCusIDCount"))
# # display(df_GetNameByDept)

# #Remove duplicate CustomerID's
# df_RemoveDuplicateCustID = df.dropDuplicates(subset=["customerID"]).dropna(subset=["customerID"])
# #display(df_RemoveDuplicateCustID)

# #Q2. List all the employees and their salary where employee's salary is more than the average salary
# df_avgSal = df_RemoveDuplicateCustID.select(F.avg("Salary").alias("avgSalary")).first()
# df_AllAvgSal = df.filter(F.col("salary")> df_avgSal.avgSalary)
# #display(df_AllAvgSal)

# #Group the sal at Department level
# df_DepartmentAvgSal = df_RemoveDuplicateCustID.select("customerID","department","name").groupBy("department").agg(F.avg("salary")),F.col("Name")
# #display(df_DepartmentAvgSal)

# #df.createOrReplaceTempView("dfdata")
# #df = spark.sql("select * from dfdata where customerID is null")
# #display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy("customerID")

df_result = (
    df
    .withColumn(
        "duplicate_count",
        F.count("*").over(w)
    )
    .withColumn(
        "rescue_reason",
        F.when(
            F.col("customerID").isNull(),
            F.lit("NULL_CUSTOMER_ID")
        )
        .when(
            F.col("name").isNull(),
            F.lit("NULL_NAME")
        )
        .when(
            F.col("salary").isNull(),
            F.lit("NULL_SALARY")
        )
        .when(
            F.col("duplicate_count") > 1,
            F.lit("DUPLICATE_CUSTOMER")
        )        
    )
    .withColumn(
        "customerID_list",
        F.sort_array(
            F.collect_set("id").over(w)
        )
    )
)

display(df_result)